<a href="https://colab.research.google.com/github/jrhumberto/2026-mestrado-pep/blob/main/etl_eleitoral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook para extração de bases - Dimensão Eleitoral


>Trata-se de notebook para extração de dados provenientes das eleições do TSE dos anos de 2018 e 2022.


**Responsável/Ano**: Humberto Bezerra de M. Júnior - 2026

Observação: Agradecimentos aos idealizadores do pacote abaixo:

* electionsBR

## Fase 0 - Etapa Inicial: Instalação de Pacotes globais

In [ ]:
install.packages("vctrs")

if (!require("devtools")) install.packages("devtools")
devtools::install_github("silvadenisson/electionsBR", force = T)

install.packages("readr")
install.packages("gt")
install.packages("flextable")
install.packages("officer")

library(vctrs) #
library(electionsBR)
library(dplyr)
library(readr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: devtools

Loading required package: usethis




openssl    (2.3.4  -> 2.3.5   ) [CRAN]
triebeard  (NA     -> 0.4.1   ) [CRAN]
httpcode   (NA     -> 0.3.0   ) [CRAN]
urltools   (NA     -> 1.7.3.1 ) [CRAN]
cpp11      (0.5.2  -> 0.5.3   ) [CRAN]
vroom      (1.6.7  -> 1.7.0   ) [CRAN]
httr       (1.4.7  -> 1.4.8   ) [CRAN]
fs         (1.6.6  -> 1.6.7   ) [CRAN]
crul       (NA     -> 1.6.0   ) [CRAN]
readr      (2.1.6  -> 2.2.0   ) [CRAN]
osfr       (NA     -> 0.2.9   ) [CRAN]
data.table (1.18.0 -> 1.18.2.1) [CRAN]
dplyr      (1.1.4  -> 1.2.0   ) [CRAN]


Installing 13 packages: openssl, triebeard, httpcode, urltools, cpp11, vroom, httr, fs, crul, readr, osfr, data.table, dplyr

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)




## Fase 1 - Busca de resultados de Eleições 2014, 2018 e 2022

In [ ]:
library(dplyr)
library(readr)
library(electionsBR)

anos <- c(2014, 2018, 2022)
cols_interesse <- c("ANO_ELEICAO", "NR_TURNO", "SG_UF", "SG_UE","NM_UE","CD_CARGO","DS_CARGO","NR_CANDIDATO",
                    "NM_CANDIDATO","NM_URNA_CANDIDATO", "TP_AGREMIACAO","NR_PARTIDO","SG_PARTIDO","NM_PARTIDO",
                    "NM_FEDERACAO","SG_FEDERACAO","DS_COMPOSICAO_FEDERACAO", "NM_COLIGACAO",
                    "DS_COMPOSICAO_COLIGACAO", "DS_SIT_TOT_TURNO")

# Criar o diretório 'data' ,caso não exista
dir.create("data", showWarnings = FALSE)

# Define função de criação de tabelas
criar_tabelas_eleicoes <- function(ano, cols_interesse, output_dir = "data") {
  # Baixa os dados
  dfe <- elections_tse(year = ano, type = "candidate")

  # Filtra e seleciona colunas
  tabela_eleicoes <- dfe %>%
    filter(DS_CARGO == "GOVERNADOR") %>%
    filter(DS_SIT_TOT_TURNO == "ELEITO") %>%
    select(any_of(cols_interesse)) %>%
    # Modifica se recebeu lixo nulo '#NULO#'
    mutate(across(where(is.character), ~na_if(., "#NULO#")))


  # Salva como CSV com o ano no nome
  nome_arquivo <- paste0(output_dir, "/", "df_eleicoes_", ano, ".csv")
  write_csv(tabela_eleicoes, nome_arquivo)

  message("Arquivo salvo: ", nome_arquivo)
}

# Para cada ano (2018 ou 2022) cria a tabela respectiva por ano no diretório 'data'
for (ano in anos) {
  criar_tabelas_eleicoes(ano, cols_interesse, "data")
}

### 1.1 Dicionário de Dados das Tabelas Eleitorais

| Campo                      | Descrição                                                                 |
| :------------------------- | :------------------------------------------------------------------------ |
| `ANO_ELEICAO`              | Ano da eleição.                                                           |
| `NR_TURNO`                 | Número do turno (1 para primeiro turno, 2 para segundo turno).             |
| `SG_UF`                    | Sigla da Unidade Federativa (Estado).                                     |
| `SG_UE`                    | Sigla da Unidade Eleitoral (Geralmente a mesma da UF para Governador).    |
| `NM_UE`                    | Nome da Unidade Eleitoral.                                                |
| `CD_CARGO`                 | Código do cargo disputado.                                                |
| `DS_CARGO`                 | Descrição do cargo (e.g., "GOVERNADOR").                                 |
| `NR_CANDIDATO`             | Número do candidato.                                                      |
| `NM_CANDIDATO`             | Nome completo do candidato.                                               |
| `NM_URNA_CANDIDATO`        | Nome do candidato que aparece na urna eletrônica.                         |
| `TP_AGREMIACAO`            | Tipo de agremiação (Partido ou Federação).                                |
| `NR_PARTIDO`               | Número do partido.                                                        |
| `SG_PARTIDO`               | Sigla do partido.                                                         |
| `NM_PARTIDO`               | Nome do partido.                                                          |
| `NM_FEDERACAO`             | Nome da federação, se aplicável.                                          |
| `SG_FEDERACAO`             | Sigla da federação, se aplicável.                                         |
| `DS_COMPOSICAO_FEDERACAO`  | Descrição da composição da federação.                                     |
| `NM_COLIGACAO`             | Nome da coligação, se aplicável.                                          |
| `DS_COMPOSICAO_COLIGACAO`  | Descrição da composição da coligação.                                     |
| `DS_SIT_TOT_TURNO`         | Situação do candidato no total de turnos (e.g., "ELEITO", "NÃO ELEITO"). |


### 1.2 Situações de Eleições Sumplementares

**Tabela – Estados com eleição suplementar para governador no Brasil (2014–2026)**

| Ano | Estado    | Motivo                                                                 | Data da eleição                              |
|-----|-----------|------------------------------------------------------------------------|-----------------------------------------------|
| 2017 | Amazonas  | Cassação do governador José Melo por compra de votos na eleição de 2014 | 6 ago. 2017                                   |
| 2018 | Tocantins | Cassação do governador Marcelo Miranda por abuso de poder econômico     | 3 jun. 2018 (1º turno) / 24 jun. 2018 (2º turno) |

**Fonte:** Elaboração própria a partir de dados do Tribunal Superior Eleitoral (TSE).


## Fase 2 - Triagem dos campos úteis
- SG_UF
- ANO_ELEICAO
- DS_CARGO
- NM_CANDIDATO
- SG_PARTIDO


In [ ]:
library(dplyr)
library(readr)

# Define a função para combinar os arquivos CSV
combinar_e_ordenar_eleicoes <- function(output_dir = "data/", output_filename = "eleicoes_2014_2018_2022.csv") {

  # Colunas desejadas na ordem especificada
  cols_finais <- c("SG_UF", "ANO_ELEICAO", "DS_CARGO", "NM_CANDIDATO", "SG_PARTIDO")

  # Lista como Buffer para armazenar os dataframes de 2018 e 2022
  lista_df <- list()
  anos <- c(2014, 2018, 2022)

  for (ano in anos) {
    file_path <- paste0(output_dir, "df_eleicoes_", ano, ".csv")
    if (file.exists(file_path)) {
      df_ano <- read_csv(file_path, show_col_types = FALSE)
      lista_df[[as.character(ano)]] <- df_ano
    } else {
      message(paste("Arquivo não encontrado para o ano", ano, ":", file_path))
    }
  }

  # Combina todos os dataframes da lista
  if (length(lista_df) > 0) {
    df_combinado <- bind_rows(lista_df)

    # Seleciona e reordena as colunas e ordena por SG_UF
    df_final <- df_combinado %>%
      select(all_of(cols_finais)) %>%

    # Eleições suplementares em AM 2017 e TO 2018
    mutate(
      DS_CARGO = if_else(
        ANO_ELEICAO == 2014 & NM_CANDIDATO == "JOSÉ MELO DE OLIVEIRA", "GOVERNADOR (CASSADO)",
        DS_CARGO
      ) ) %>%
    mutate(
      ANO_ELEICAO = if_else(
        ANO_ELEICAO == 2014 &NM_CANDIDATO == "AMAZONINO ARMANDO MENDES", 2017,
        ANO_ELEICAO
      ) ) %>%
    mutate(
      DS_CARGO = if_else(
        ANO_ELEICAO == 2014 & NM_CANDIDATO == "MARCELO DE CARVALHO MIRANDA", "GOVERNADOR (CASSADO)",
        DS_CARGO
      ) ) %>%
    mutate(
      ANO_ELEICAO = if_else(
        ANO_ELEICAO == 2014 & NM_CANDIDATO == "MAURO CARLESSE", 2018,  ANO_ELEICAO
      )) %>%

    arrange(SG_UF)
    # Salva o dataframe combinado e ordenado
    final_output_path <- paste0(output_dir, output_filename)
    write_csv(df_final, final_output_path)

    message("Arquivo combinado salvo: ", final_output_path)
    return(df_final)
  } else {
    message("Nenhum arquivo CSV encontrado para combinar.")
    return(NULL)
  }
}

# Chama a função para combinar os arquivos
eleicoes_2014_2018_2022 <- combinar_e_ordenar_eleicoes()

## Fase 3 - Criação da variável austeridade, será 1 se o governador eleito for dos seguintes partidos:
- NOVO
- PL
- PSDB
- DEM
- UNIÂO
- PSD
- REPUBLICANOS

In [ ]:
# Defina as siglas dos partidos que representam 'austeridade'
siglas_austeridade_fiscal <- c("NOVO", "PL", "PSDB", "REPUBLICANOS",
                               "DEM", "UNIÃO", "PSD")

# Crie a nova coluna 'austeridade'
eleicoes_2014_2018_2022_austeridade <- eleicoes_2014_2018_2022 %>%
  mutate(austeridade = ifelse(SG_PARTIDO %in% siglas_austeridade_fiscal, 1, 0))


# Salve o resultado em um novo arquivo CSV
output_path_austeridade <- "data/eleicoes_2014_2018_2022_austeridade.csv"
write_csv(eleicoes_2014_2018_2022_austeridade, output_path_austeridade)

message(paste0("Arquivo com coluna 'austeridade' salvo em: ", output_path_austeridade))


O dataframe `eleicoes_2018_2022_austeridade` foi criado com a nova coluna `austeridade` e salvo em `data/eleicoes_2018_2022_austeridade.csv`.



---



## Fase 4 - Criação das variáveis reeleito e igual partido do Governador anterior.


In [ ]:
head(eleicoes_2014_2018_2022_austeridade,30)

In [ ]:
# Crie um dataset apenas dos pleitos de 2018 e 2022
df_2018_2022 <- eleicoes_2014_2018_2022_austeridade %>%
  # A. Organiza para garantir que os anos estejam na sequência correta por estado
  arrange(SG_UF, ANO_ELEICAO) %>%

  # B. Agrupar por UF
  group_by(SG_UF) %>%

  # C. Criar as colunas comparando com o registro anterior - lag
  mutate(
    reeleito = if_else(NM_CANDIDATO == lag(NM_CANDIDATO), 1, 0),
    mesmopartido = if_else(
      # Esse trecho vai permitir PMDB==MDB ou MDB=PMDB pois MDB==MDB
      substr(SG_PARTIDO, nchar(SG_PARTIDO) - 2, nchar(SG_PARTIDO)) ==
      substr(lag(SG_PARTIDO), nchar(lag(SG_PARTIDO)) - 2, nchar(lag(SG_PARTIDO))),
      1, 0
    )
  ) %>%

  # D. Desagrupa
  ungroup() %>%

  # E. Filtragem para remover o ano de 2014 e eleição suplementar do AM em 2017
  filter(ANO_ELEICAO != 2014) %>%
  filter(ANO_ELEICAO != 2017) %>%

  # F. Remover Mauro Carlesse que está pela eleição suplementar de 24/06/2018
  filter(!(NM_CANDIDATO == 'MAURO CARLESSE' & reeleito == 0))



# Salve o resultado em um novo arquivo CSV
output_path_austeridade <- "data/eleitoral_uf_2018_2022_austeridade_reeleito_mesmopartido.csv"
write_csv(df_2018_2022, output_path_austeridade)

message(paste0("Arquivo com coluna 'austeridade' salvo em: ", output_path_austeridade))

In [ ]:
head(df_2018_2022, 30)

---

## Fase 5 - Criação de tabelas para dissertação (parte opcional)

### Funções Genéricas e Pacotes



In [ ]:
library(flextable)
library(officer)
library(dplyr)
library(tibble)



monta_tabela <- function(dados, largs, titulo, fonte, path_my, name) {

  # Cria diretorio
  dir.create(path_my, showWarnings = FALSE)

  if ("Ano" %in% names(dados)) {
      dados <- dados %>% arrange(Ano) %>% mutate(Ano = as.character(Ano))
  }

  # Estilo ABNT de bordas
  t <- flextable(dados) %>%
    border_remove() %>%
    hline_top(part = "header", border = fp_border(color="black", width = 1.5)) %>%
    hline_bottom(part = "header", border = fp_border(color="black", width = 1)) %>%
    hline_bottom(part = "body", border = fp_border(color="black", width = 1.5)) %>%

    # Legenda (Título) em tamanho 12 e Fonte
    set_caption(caption = titulo, style = "Table Caption") %>%
    add_footer_lines(fonte) %>%

    # Fonte e tamanho 10
    font(fontname = "Times New Roman", part = "all") %>%
    fontsize(size = 10, part = "body") %>%
    fontsize(size = 10, part = "header") %>%
    fontsize(size = 10, part = "footer") %>%

    # Fixa larguras das colunas e acerta a borda
    width(j = seq_along(largs), width = largs) %>%
    fix_border_issues()

  # Guarde
  save_as_docx(t, path = paste0(path_my,'/', name, ".docx"))

  return(t)
}

### 5.1 Tabela com fusões e incorporações de partidos políticos no Brasil

In [ ]:
brasil_siglas <- tibble(
  Ano = c(2022, 2023, 2019, 2019, 2019, 2023, 2023),

  Tipo = c("Fusão","Fusão","Incorporação","Incorporação","Incorporação","Incorporação","Incorporação"),

  `Partidos envolvidos` = c("PSL + DEM", "PTB + Patriota", "PRP → Patriota", "PPL → PCdoB",
     "PHS → Podemos", "PROS → Solidariedade", "PSC → Podemos"
  ),

  Resultado = c("União Brasil","PRD","Patriota","PCdoB","Podemos","Solidariedade","Podemos"  ),

  `Data da decisão/homologação` = c("08 fev. 2022","09 nov. 2023","28 mar. 2019","28 maio 2019",
    "19 set. 2019", "14 fev. 2023","14 dez. 2023"
  )
)
w_brasil_siglas <- c(0.5, 1.0, 1.5, 1.0, 1.5)

In [ ]:
# Parametros: Tibble + VectorWidths+ Titulo + Fonte + Path + Name
monta_tabela(
  brasil_siglas, w_brasil_siglas,
  "Tabela 1 – Fusões e incorporações de partidos políticos no Brasil - 2019 até 2024",
  "Fonte: Elaboração própria a partir de dados eleitorais - TSE (2024).",
  'tabelas', 't1_tabela_siglas_brasil'
)

### 5.2 Tabela com os governadores eleitos em 2014, 2018 e 2022


In [ ]:

# Dados do Dataframe resgatados para preparo
dados_preparados <- eleicoes_2014_2018_2022_austeridade %>%
  select(
    UF = SG_UF,
    Ano = ANO_ELEICAO,
    Governador = NM_CANDIDATO,
    Partido = SG_PARTIDO,
  )

# Elabora Uma tabela com 4 colunas
tabela_word <- flextable(dados_preparados) %>%
  set_header_labels(
    Governador = "Governador(a)"
  ) %>%

  # Retiro vírgula do ano
  colformat_double(j = "Ano", big.mark = "", digits = 0) %>%

  # Ajuste colunas (Total aprox 6.5 polegadas para A4)
  width(j = 1, width = 0.5) %>% # UF
  width(j = 2, width = 0.5) %>% # Ano
  width(j = 3, width = 2.5) %>% # Governador
  width(j = 4, width = 1.5) %>% # Partido

  # Estilo ABNT: Limpar e colocar linhas horizontais
  border_remove() %>%
  hline_top(part = "header", border = fp_border(color="black", width = 1.5)) %>%
  hline_bottom(part = "header", border = fp_border(color="black", width = 1)) %>%
  hline_bottom(part = "body", border = fp_border(color="black", width = 1.5)) %>%

  # Alinhamento (UF e Partido à esquerda, o resto centralizado)
  align(j = c(1, 3, 4), align = "left", part = "all") %>%
  align(j = c(2), align = "center", part = "all") %>%

  # Fonte e Tamanho
  font(fontname = "Times New Roman", part = "all") %>%
  fontsize(size = 10, part = "body") %>%
  fontsize(size = 10, part = "header") %>%
  fontsize(size = 10, part = "footer") %>%

  # Legenda e Fonte (ABNT)
  set_caption(caption = "Tabela 2 – Governadores Eleitos no Brasil - 2014, 2018 e 2022") %>%
  add_footer_lines("Fonte: Elaborado pelo autor com base em dados do TSE (2024).") %>%
  add_footer_lines("Nota: Governador de TO venceu 2 pleitos em 2018: Eleição Suplementar em 24/06 e reeleito em Outubro.") %>%
  fix_border_issues()

# Guarda em arquivo
save_as_docx(tabela_word, path = "tabelas/t2_tabela_governadores.docx")

# Visualizar no R
print(tabela_word)

### 5.3 Tabela com os pleitos eleitorais de 2018 e 2022 e seus indicadores calculados

In [ ]:
# Dados do Dataframe resgatados para preparo
dados_preparados <- df_2018_2022 %>%
  select(
    UF = SG_UF,
    Ano = ANO_ELEICAO,
    Austeridade = austeridade,
    Reeleicao = reeleito,
    Mesmo = mesmopartido
  )

# Elabora Uma tabela com 6 colunas
tabela_word <- flextable(dados_preparados) %>%
  #merge_v(j = 1) %>%
  set_header_labels(
    Reeleicao = "Reeleito",
    Mesmo = "Mesmo Partido do Governador anterior"
  ) %>%

  # Retiro vírgula do ano
  colformat_double(j = "Ano", big.mark = "", digits = 0) %>%

  # Ajuste colunas (Total aprox 6.5 polegadas para A4)
  width(j = 1, width = 0.5) %>% # UF
  width(j = 2, width = 0.5) %>% # Ano
  width(j = 3, width = 1.0) %>% # Austeridade
  width(j = 4, width = 1.0) %>% # Reeleição
  width(j = 5, width = 1.5) %>% # Sucessão

  # Padrão ABNT: Apenas linhas horizontais, não coloque verticais
  border_remove() %>%
  hline_top(part = "header", border = fp_border(color="black", width = 1.5)) %>%
  hline_bottom(part = "header", border = fp_border(color="black", width = 1)) %>%
  hline_bottom(part = "body", border = fp_border(color="black", width = 1.5)) %>%

  # Alinhamento (UF e Partido à esquerda, o resto centralizado)
  align(j = c(1), align = "left", part = "all") %>%
  align(j = c(2,3, 4, 5), align = "center", part = "all") %>%

  # Fonte e Tamanho
  font(fontname = "Times New Roman", part = "all") %>%
  fontsize(size = 10, part = "body") %>%
  fontsize(size = 10, part = "header") %>%
  fontsize(size = 10, part = "footer") %>%

  # Legenda e Fonte (ABNT)
  set_caption(caption = "Tabela 3 – Variáveis eleitorais: austeridade,  se foi reeleito e se o vencedor é do mesmo partido do Governador anterior - Brasil (Eleições 2018 e 2022)") %>%
  add_footer_lines("Fonte: Elaborado pelo autor com base em dados do TSE (2024).") %>%
  fix_border_issues()

# Guarda
save_as_docx(tabela_word, path = "tabelas/t3_tabela_indicadores.docx")



---